# Análise de dados Industriais

#### Importações

In [177]:
import pandas as pd
import numpy as np
from datetime import datetime


In [176]:
# Função para detectar outliers
def listar_outliers(df, coluna):
    """
    Retorna um novo DataFrame contendo apenas os outliers da coluna especificada.
    Utiliza o método do Intervalo Interquartil (IQR).
    """
    # 1. Calcular os quartis e o IQR
    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    IQR = Q3 - Q1

    # 2. Definir os limites inferior e superior
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # 3. Filtrar o DataFrame para retornar apenas os outliers
    outliers = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]

    return outliers

### Extração e Tratamento dos dados sensores.json

In [279]:
df_sensores = pd.read_json('/content/sensores.json')
df_medicoes = pd.DataFrame(dict(df_sensores['medicoes'])).T

df_sensores = df_sensores.drop(columns=['medicoes'], axis=1)

df_sensores = pd.concat([df_sensores, df_medicoes], axis=1)

# O df_sensores apresentou inconsistencia na quantidade de registro da serie 'temperaturas'
# Etapa 1 verificar a quantidade de registos nulos
# print(df_sensores.isna().sum())
# Verificado 6 registros nulos
# Etapa 2 substituir os valores nulos pela mediana
df_sensores['temperatura'] = df_sensores['temperatura'].fillna(df_sensores['temperatura'].median())

# Verificar duplicados
# print(f'Duplicados: {df_sensores.duplicated().sum()}')
# verificado 4 duplicados
# Etapa 3 excluir duplicados
df_sensores = df_sensores.drop_duplicates()
# converter para datetime e padronizar o formato para futuro merge
df_sensores['timestamp'] = pd.to_datetime(df_sensores['timestamp']).dt.strftime('%Y-%m-%d')
# renomear a coluna timestamp para data
df_sensores = df_sensores.rename(columns={'timestamp': 'data'})
df_sensores['data'] = pd.to_datetime(df_sensores['data'])

df_sensores




,id_maquina,data,status,consumo_energia,temperatura,vibracao,pressao,velocidade
0,MAQ-03,2025-03-20,operando,11.86,68.1,2.32,5.93,1288.8
1,MAQ-10,2025-03-24,operando,13.89,60.4,1.91,4.49,1020.1
2,MAQ-10,2025-03-23,operando,9.88,65.5,2.08,4.24,1181.3
3,MAQ-02,2025-03-04,operando,13.63,66.4,2.36,5.34,1153.6
4,MAQ-07,2025-03-24,operando,14.98,70.5,2.77,5.73,1155.1
...,...,...,...,...,...,...,...,...
609,MAQ-02,2025-03-09,operando,11.03,68.3,2.20,5.66,1289.3
610,MAQ-03,2025-03-27,operando,15.25,70.1,1.88,5.68,1158.5
611,MAQ-07,2025-03-19,operando,10.36,64.8,1.89,6.01,1276.7
612,MAQ-08,2025-03-21,operando,13.10,65.5,3.20,4.43,1169.6


### Extracao e Tratamento dos dados producao.csv

In [271]:
df_producao = pd.read_csv('/content/producao.csv')

# Etapa 1 Verificar dados nulos
# print(df_producao.isna().sum())
# identificado valores nulos na 'quantidade_produzida', 'tempo_producao' e 'operador'
# Substituir valores nulos de 'quantidade_produzida', 'tempo_producao' pela mediana
df_producao['quantidade_produzida'] = df_producao['quantidade_produzida']\
                .fillna(df_producao['quantidade_produzida'].median())
df_producao['tempo_producao'] = df_producao['tempo_producao']\
                .fillna(df_producao['tempo_producao'].median())
# Substituir valores nulos de 'operador' pela mediana
df_producao['operador'] = df_producao['operador']\
                .fillna(df_producao['operador'].mode()[0])

# Etapa 2 Verificar Duplicados
# print(df_producao.duplicated().sum())
# Localizado 5 valores duplciados
df_producao = df_producao.drop_duplicates()
# Convertendo data para datetime e padronizando o formato
df_producao['data'] = pd.to_datetime(df_producao['data'], yearfirst=True, format='mixed')

# Etapa 3 Padronização dos nomes das Máquinas
# print(df_producao['maquina'].unique())
'''Removendo espaços do inicio e final com strip, alterando para caixa alta com upper \
e substituindo espaços duplo por espaço simples'''
df_producao['maquina'] = df_producao['maquina'].str.strip().str.upper().str.replace('  ', ' ')
'''Nos locais nos numeros 1 adicionei um zero e depois replace 00 por 0 para que
o padrão seja exemplo: INJETORA PLÁSTICA 01'''
df_producao['maquina'] = df_producao['maquina'].str[:-1] + '0' + df_producao['maquina'].str[-1]
df_producao['maquina'] = df_producao['maquina'].str.replace('00', '0')

# print('\n Após padronização do nome')
# print(df_producao['maquina'].unique())

# Foi identificado valores iguais a zero em tempo_producao
# print(df_producao[df_producao['tempo_producao'] == 0][['maquina', 'tempo_producao']])
# Substitui valores 0 pela mediana do tempo_producao e arendodar para duas casas decimais
df_producao['tempo_producao'] = df_producao['tempo_producao'].replace(0, df_producao['tempo_producao'].median()).round(2)

# padronizando os turnos
# print(df_producao['turno'].unique())
df_producao['turno'] = df_producao['turno'].str.strip().str.lower()
# print(df_producao['turno'].unique())

df_producao['quantidade_produzida'] = df_producao['quantidade_produzida'].astype('int')


# listar_outliers(df_producao, 'tempo_producao')

df_producao[df_producao['quantidade_produzida'] < 0]










,id_producao,data,maquina,produto,quantidade_produzida,quantidade_planejada,tempo_producao,operador,turno
6,PR0133,2025-12-03,PRENSA HIDRÁULICA 01,Suporte Metálico,-28,600,348.0,Carlos Souza,noite
63,PR0073,2025-03-07,FRESADORA 01,Carcaça Plástica,-7,700,379.0,Roberto Alves,noite
137,PR0303,2025-03-27,INJETORA PLÁSTICA 01,Eixo Automotivo,-27,480,466.0,Juliana Costa,noite
175,PR0224,2025-03-20,FRESADORA 02,Engrenagem Industrial,-5,350,416.0,Camila Rocha,noite


### Extracao e Tratamento dos dados manutencao.xlsx

In [246]:
df_manutencao = pd.read_excel('manutencao.xlsx')

# df_manutencao.isna().sum()
# Valores Nulos pela mediana
df_manutencao['tempo_parada'] = df_manutencao['tempo_parada'].fillna(df_manutencao['tempo_parada'].median())
df_manutencao['tempo_parada'] = df_manutencao['tempo_parada'].astype('int')
# df_manutencao.isna().sum()

# Verifica duplicados
# df_manutencao.duplicated().sum()

# Padronização Nomes das Maquinas
# print(df_manutencao['maquina'].unique())
'''Removendo espaços do inicio e final com strip, alterando para caixa alta com upper \
e substituindo espaços duplo por espaço simples'''
df_manutencao['maquina'] = df_manutencao['maquina'].str.strip().str.upper().str.replace('  ', ' ')
'''Nos locais nos numeros 1 adicionei um zero e depois replace 00 por 0 para que
o padrão seja exemplo: INJETORA PLÁSTICA 01'''
df_manutencao['maquina'] = df_manutencao['maquina'].str[:-1] + '0' + df_manutencao['maquina'].str[-1]
df_manutencao['maquina'] = df_manutencao['maquina'].str.replace('00', '0')
# print(df_manutencao['maquina'].unique())

# Padronizando data
df_manutencao['data'] = pd.to_datetime(df_manutencao['data'], format='mixed', yearfirst=True)

# Padronização dos tipos
# print(df_manutencao['tipo_manutencao'].unique())
df_manutencao['tipo_manutencao'] = df_manutencao['tipo_manutencao'].str.strip().str.upper().str.replace('  ', ' ')
# print(df_manutencao['tipo_manutencao'].unique())

df_manutencao



,id_manutencao,maquina,data,tipo_manutencao,motivo,tempo_parada,custo_manutencao,tecnico
0,MAN0066,SOLDA ROBOTIZADA 02,2025-06-03,CORRETIVA,Substituição de rolamento,98,824.57,Eduardo Martins
1,MAN0017,TORNO CNC 02,2025-03-29,PREVENTIVA,Ajuste de calibração,104,733.04,Vanessa Teixeira
2,MAN0010,TORNO CNC 02,2025-03-02,PREVENTIVA,Ajuste de calibração,117,774.66,Renata Dias
3,MAN0005,TORNO CNC 01,2025-03-08,PREVENTIVA,Substituição de rolamento,47,720.72,Eduardo Martins
4,MAN0087,EXTRUSORA 01,2025-03-14,PREVENTIVA,Lubrificação programada,112,553.13,Vanessa Teixeira
...,...,...,...,...,...,...,...,...
88,MAN0046,PRENSA HIDRÁULICA 02,2025-03-01,PREVENTIVA,Limpeza de filtros,75,983.41,Vanessa Teixeira
89,MAN0058,SOLDA ROBOTIZADA 01,2025-03-13,PREVENTIVA,Revisão elétrica,75,742.73,Sérgio Barbosa
90,MAN0023,FRESADORA 01,2025-03-16,PREVENTIVA,Troca de peça de desgaste,117,291.68,Vanessa Teixeira
91,MAN0007,TORNO CNC 01,2025-03-17,CORRETIVA,Troca de peça de desgaste,58,652.54,Renata Dias


### Extracao e Tratamento dos dados qualidade.parquet

In [264]:
df_qualidade = pd.read_parquet('/content/qualidade.parquet')

# Verificando e atualizando valores nulos para a mediana
df_qualidade.isna().sum()
df_qualidade['quantidade_aprovada'] = df_qualidade['quantidade_aprovada'].fillna(df_qualidade['quantidade_aprovada'].median())

# Verificando e deletando duplicados
# df_qualidade.duplicated().sum()

# Padronização Nomes das Maquinas
# print(df_qualidade['maquina'].unique())
'''Removendo espaços do inicio e final com strip, alterando para caixa alta com upper \
e substituindo espaços duplo por espaço simples'''
df_qualidade['maquina'] = df_qualidade['maquina'].str.strip().str.upper().str.replace('  ', ' ')
'''Nos locais nos numeros 1 adicionei um zero e depois replace 00 por 0 para que
o padrão seja exemplo: INJETORA PLÁSTICA 01'''
df_qualidade['maquina'] = df_qualidade['maquina'].str[:-1] + '0' + df_qualidade['maquina'].str[-1]
df_qualidade['maquina'] = df_qualidade['maquina'].str.replace('00', '0')
# print(df_qualidade['maquina'].unique())

df_qualidade['quantidade_aprovada'] = df_qualidade['quantidade_aprovada'].astype('int')
df_qualidade['meta_qualidade'] = df_qualidade['meta_qualidade'].astype('int')

df_qualidade

,id_producao,maquina,produto,quantidade_inspecionada,quantidade_aprovada,quantidade_rejeitada,indice_qualidade,meta_qualidade
0,PR0222,TORNO CNC 01,Suporte Metálico,256,254,2,99.22,95
1,PR0236,PRENSA HIDRÁULICA 02,Engrenagem Industrial,88,87,1,98.86,95
2,PR0322,PRENSA HIDRÁULICA 01,Componente Hidráulico,153,136,7,95.10,95
3,PR0163,PRENSA HIDRÁULICA 02,Engrenagem Industrial,117,110,7,94.02,95
4,PR0266,SOLDA ROBOTIZADA 02,Suporte Metálico,258,225,33,87.21,95
...,...,...,...,...,...,...,...,...
231,PR0160,PRENSA HIDRÁULICA 01,Suporte Metálico,92,86,6,93.48,95
232,PR0233,FRESADORA 02,Engrenagem Industrial,227,213,14,93.83,95
233,PR0277,SOLDA ROBOTIZADA 02,Engrenagem Industrial,99,79,20,79.80,95
234,PR0070,TORNO CNC 01,Carcaça Plástica,162,149,13,91.98,95
